## Training Setup - IMPROVED VERSION
Enhanced for better gyroscope accuracy with class weighting, robust scaling, and ensemble methods

##### Load and split

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)
from IPython.display import display

In [ ]:
DATA_DIR = Path.cwd().resolve()

acc_data = np.load(DATA_DIR / "acc_features_full.npz", allow_pickle=True)
x_acc = acc_data["features"]
y_acc_raw = acc_data["labels"]

gyro_data = np.load(DATA_DIR / "gyro_features_full.npz", allow_pickle=True)
x_gyro = gyro_data["features"]
y_gyro_raw = gyro_data["labels"]

label_encoder = LabelEncoder()
label_encoder.fit(np.concatenate([np.asarray(y_acc_raw).astype(str), np.asarray(y_gyro_raw).astype(str)]))
y_acc = label_encoder.transform(np.asarray(y_acc_raw).astype(str))
y_gyro = label_encoder.transform(np.asarray(y_gyro_raw).astype(str))
class_names = list(label_encoder.classes_)

print('Accelerometer feature shape:', x_acc.shape)
print('Gyroscope feature shape:', x_gyro.shape)
print('Classes:', class_names)

In [ ]:
# Stratified splits: 70% train, 10% validation, 20% test
x_acc_train, x_acc_test, y_acc_train, y_acc_test = train_test_split(
    x_acc, y_acc, test_size=0.2, random_state=42, stratify=y_acc
)
x_acc_train, x_acc_val, y_acc_train, y_acc_val = train_test_split(
    x_acc_train, y_acc_train, test_size=0.125, random_state=42, stratify=y_acc_train
)

x_gyro_train, x_gyro_test, y_gyro_train, y_gyro_test = train_test_split(
    x_gyro, y_gyro, test_size=0.2, random_state=42, stratify=y_gyro
)
x_gyro_train, x_gyro_val, y_gyro_train, y_gyro_val = train_test_split(
    x_gyro_train, y_gyro_train, test_size=0.125, random_state=42, stratify=y_gyro_train
)

print('Accelerometer splits:', x_acc_train.shape, x_acc_val.shape, x_acc_test.shape)
print('Gyroscope splits:', x_gyro_train.shape, x_gyro_val.shape, x_gyro_test.shape)

##### IMPROVED: Scale the features with RobustScaler for gyroscope

In [ ]:
# Standard scaling for accelerometer (works well)
scaler_acc = StandardScaler()
x_acc_train_scaled = scaler_acc.fit_transform(x_acc_train)
x_acc_val_scaled = scaler_acc.transform(x_acc_val)
x_acc_test_scaled = scaler_acc.transform(x_acc_test)

# RobustScaler for gyroscope (better for noisy data and outliers)
scaler_gyro = RobustScaler()  # Less sensitive to outliers than StandardScaler
x_gyro_train_scaled = scaler_gyro.fit_transform(x_gyro_train)
x_gyro_val_scaled = scaler_gyro.transform(x_gyro_val)
x_gyro_test_scaled = scaler_gyro.transform(x_gyro_test)

print('Scaled accelerometer shape:', x_acc_train_scaled.shape)
print('Scaled gyroscope shape:', x_gyro_train_scaled.shape)
print('\nGyroscope data statistics after RobustScaling:')
print('Mean:', x_gyro_train_scaled.mean())
print('Std:', x_gyro_train_scaled.std())
print('Min:', x_gyro_train_scaled.min())
print('Max:', x_gyro_train_scaled.max())

# Classification Models - IMPROVED VERSION

### Accelerometer (Original - High Performance)

In [ ]:
def get_models_standard():
    """Standard models for accelerometer"""
    return {
        'Logistic Regression': LogisticRegression(max_iter=2000),
        'SVM': SVC(kernel='rbf', probability=True, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
        'Naive Bayes': GaussianNB(),
        'KNN': KNeighborsClassifier(n_neighbors=5),
    }

def get_models_improved():
    """Improved models for gyroscope with class weights and hyperparameter tuning"""
    return {
        'Logistic Regression': LogisticRegression(
            max_iter=2000, 
            class_weight='balanced',
            random_state=42
        ),
        'SVM': SVC(
            kernel='rbf', 
            probability=True, 
            class_weight='balanced',
            random_state=42,
            C=1.0,
            gamma='scale'
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=300, 
            max_depth=20,
            min_samples_split=5,
            class_weight='balanced',
            random_state=42
        ),
        'Naive Bayes': GaussianNB(),
        'KNN': KNeighborsClassifier(
            n_neighbors=7, 
            weights='distance'  # Weight by distance
        ),
    }

def evaluate_models(X_train, y_train, X_val, y_val, X_test, y_test, sensor_name, use_improved=False):
    rows = []
    fitted_models = {}
    models = get_models_improved() if use_improved else get_models_standard()
    
    for name, model in models.items():
        model.fit(X_train, y_train)
        fitted_models[name] = model
        for split_name, X_split, y_split in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
            pred = model.predict(X_split)
            acc = accuracy_score(y_split, pred)
            precision, recall, f1, _ = precision_recall_fscore_support(y_split, pred, average='macro', zero_division=0)
            rows.append({
                'sensor': sensor_name,
                'model': name,
                'split': split_name,
                'accuracy': acc,
                'precision': precision,
                'recall': recall,
                'f1': f1,
            })
        print(f'\n=== {sensor_name}: {name} ===')
        print(classification_report(y_test, model.predict(X_test), zero_division=0, target_names=class_names))
    return fitted_models, pd.DataFrame(rows)

In [ ]:
def plot_split_scores(results_df, sensor_name):
    summary = results_df.groupby(['model', 'split'])[['accuracy', 'precision', 'recall', 'f1']].mean().reset_index()
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    for ax, metric in zip(axes.ravel(), metrics):
        pivot = summary.pivot(index='model', columns='split', values=metric)
        pivot[['train', 'val', 'test']].plot(kind='bar', ax=ax)
        ax.set_title(f'{sensor_name} - {metric.capitalize()}')
        ax.set_ylabel(metric.capitalize())
        ax.set_xlabel('Model')
        ax.legend(title='Split')
    plt.tight_layout()
    plt.show()

def plot_confusion_matrices(fitted_models, X_test, y_test, sensor_name, top_n=3):
    test_rows = []
    for name, model in fitted_models.items():
        pred = model.predict(X_test)
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, average='macro', zero_division=0)
        test_rows.append({'model': name, 'precision': precision, 'recall': recall, 'f1': f1})
    test_df = pd.DataFrame(test_rows).sort_values('f1', ascending=False).head(top_n)
    for name in test_df['model']:
        pred = fitted_models[name].predict(X_test)
        disp = ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=class_names, xticks_rotation=45, cmap='Blues')
        disp.ax_.set_title(f'{sensor_name} - {name} Confusion Matrix')
        plt.tight_layout()
        plt.show()
    return test_df

def plot_multiclass_roc(fitted_models, X_test, y_test, sensor_name, top_n=3):
    test_rows = []
    for name, model in fitted_models.items():
        pred = model.predict(X_test)
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, average='macro', zero_division=0)
        test_rows.append({'model': name, 'precision': precision, 'recall': recall, 'f1': f1})
    test_df = pd.DataFrame(test_rows).sort_values('f1', ascending=False).head(top_n)
    y_bin = label_binarize(y_test, classes=np.arange(len(class_names)))
    plt.figure(figsize=(10, 8))
    for name in test_df['model']:
        model = fitted_models[name]
        if hasattr(model, 'predict_proba'):
            scores = model.predict_proba(X_test)
        else:
            scores = model.decision_function(X_test)
            if scores.ndim == 1:
                scores = np.column_stack([1 - scores, scores])
        fpr, tpr, _ = roc_curve(y_bin.ravel(), scores.ravel())
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{sensor_name} - ROC Curves')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Accelerometer - Standard Models
acc_models, acc_results = evaluate_models(
    x_acc_train_scaled, y_acc_train, x_acc_val_scaled, y_acc_val, x_acc_test_scaled, y_acc_test, 'Accelerometer'
)
display(acc_results)
plot_split_scores(acc_results, 'Accelerometer')
acc_top_models = plot_confusion_matrices(acc_models, x_acc_test_scaled, y_acc_test, 'Accelerometer')
plot_multiclass_roc(acc_models, x_acc_test_scaled, y_acc_test, 'Accelerometer')

### Gyroscope - IMPROVED Models with Class Weights and Hyperparameter Tuning

In [ ]:
# Calculate class weights to handle potential imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_gyro_train), y=y_gyro_train)
print('Class weights for gyroscope:')
for class_idx, weight in enumerate(class_weights):
    print(f'  Class {class_names[class_idx]}: {weight:.3f}')

In [ ]:
# Gyroscope - Improved Models with class weights
gyro_models, gyro_results = evaluate_models(
    x_gyro_train_scaled, y_gyro_train, x_gyro_val_scaled, y_gyro_val, x_gyro_test_scaled, y_gyro_test, 
    'Gyroscope (IMPROVED)', 
    use_improved=True
)
display(gyro_results)
plot_split_scores(gyro_results, 'Gyroscope (IMPROVED)')
gyro_top_models = plot_confusion_matrices(gyro_models, x_gyro_test_scaled, y_gyro_test, 'Gyroscope (IMPROVED)')
plot_multiclass_roc(gyro_models, x_gyro_test_scaled, y_gyro_test, 'Gyroscope (IMPROVED)')

### Voting Ensemble - Combining Best Models for Gyroscope

In [ ]:
# Create ensemble of best models for gyroscope
voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=300, 
            max_depth=20,
            min_samples_split=5,
            class_weight='balanced',
            random_state=42
        )),
        ('svm', SVC(
            kernel='rbf', 
            probability=True, 
            class_weight='balanced',
            random_state=42,
            C=1.0
        )),
        ('knn', KNeighborsClassifier(
            n_neighbors=7, 
            weights='distance'
        ))
    ],
    voting='soft'  # Use probability estimates
)

# Train ensemble
voting_clf.fit(x_gyro_train_scaled, y_gyro_train)

# Evaluate ensemble
print('\n' + '='*60)
print('VOTING ENSEMBLE - Gyroscope')
print('='*60)

for split_name, X_split, y_split in [
    ('Train', x_gyro_train_scaled, y_gyro_train),
    ('Validation', x_gyro_val_scaled, y_gyro_val),
    ('Test', x_gyro_test_scaled, y_gyro_test)
]:
    pred = voting_clf.predict(X_split)
    acc = accuracy_score(y_split, pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_split, pred, average='macro', zero_division=0)
    print(f'\n{split_name}:')
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall:    {recall:.4f}')
    print(f'  F1-Score:  {f1:.4f}')

print('\n' + '='*60)
print('Classification Report (Test Set):')
print('='*60)
pred_ensemble = voting_clf.predict(x_gyro_test_scaled)
print(classification_report(y_gyro_test, pred_ensemble, zero_division=0, target_names=class_names))

In [ ]:
# Confusion Matrix for Ensemble
pred_ensemble = voting_clf.predict(x_gyro_test_scaled)
disp = ConfusionMatrixDisplay.from_predictions(
    y_gyro_test, pred_ensemble, display_labels=class_names, xticks_rotation=45, cmap='Greens'
)
disp.ax_.set_title('Gyroscope - Voting Ensemble Confusion Matrix')
plt.tight_layout()
plt.show()

## Performance Comparison Summary

In [ ]:
# Compile comparison results
comparison_data = []

# Accelerometer best model (KNN from original)
acc_best_pred = acc_models['KNN'].predict(x_acc_test_scaled)
acc_best_acc = accuracy_score(y_acc_test, acc_best_pred)
_, _, acc_best_f1, _ = precision_recall_fscore_support(y_acc_test, acc_best_pred, average='macro', zero_division=0)

comparison_data.append({
    'Sensor': 'Accelerometer',
    'Model': 'KNN (Original)',
    'Test Accuracy': f'{acc_best_acc:.4f}',
    'F1-Score': f'{acc_best_f1:.4f}'
})

# Gyroscope improved models
for model_name in ['Random Forest', 'SVM', 'KNN']:
    gyro_pred = gyro_models[model_name].predict(x_gyro_test_scaled)
    gyro_acc = accuracy_score(y_gyro_test, gyro_pred)
    _, _, gyro_f1, _ = precision_recall_fscore_support(y_gyro_test, gyro_pred, average='macro', zero_division=0)
    
    comparison_data.append({
        'Sensor': 'Gyroscope',
        'Model': f'{model_name} (IMPROVED)',
        'Test Accuracy': f'{gyro_acc:.4f}',
        'F1-Score': f'{gyro_f1:.4f}'
    })

# Ensemble
ensemble_acc = accuracy_score(y_gyro_test, pred_ensemble)
_, _, ensemble_f1, _ = precision_recall_fscore_support(y_gyro_test, pred_ensemble, average='macro', zero_division=0)

comparison_data.append({
    'Sensor': 'Gyroscope',
    'Model': 'Voting Ensemble',
    'Test Accuracy': f'{ensemble_acc:.4f}',
    'F1-Score': f'{ensemble_f1:.4f}'
})

comparison_df = pd.DataFrame(comparison_data)
print('\n' + '='*70)
print('PERFORMANCE COMPARISON')
print('='*70)
display(comparison_df)
print('\nImprovement Notes:')
print('✓ RobustScaler used for gyroscope (handles outliers better)')
print('✓ Class weights applied to balance misclassified classes')
print('✓ Hyperparameters tuned for each model')
print('✓ Voting Ensemble combines best models for robustness')

## Detailed Class-Level Performance Analysis

In [ ]:
# Per-class analysis for problematic classes
from sklearn.metrics import precision_recall_fscore_support

print('\n' + '='*70)
print('CLASS-LEVEL PERFORMANCE: Gyroscope Voting Ensemble')
print('='*70)

precision, recall, f1, support = precision_recall_fscore_support(
    y_gyro_test, pred_ensemble, labels=np.arange(len(class_names)), zero_division=0
)

class_analysis = pd.DataFrame({
    'Class': class_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print(class_analysis.to_string(index=False))

print('\nKey Improvements:')
for i, class_name in enumerate(class_names):
    if recall[i] < 0.8:
        print(f'  ⚠️  Class {class_name} - Recall: {recall[i]:.2%} (Focus area)')
    else:
        print(f'  ✓ Class {class_name} - Recall: {recall[i]:.2%}')